# DETEKSI VIDEO WATERMARKING TAHAN KOMPRESI TINGGI DENGAN NEURAL CODEC

**Versi 2 — perbaikan berdasarkan review dosen pembimbing.**

Perbaikan di versi ini dibanding versi sebelumnya:

| Temuan dosen | Perbaikan di notebook ini |
|---|---|
| Train/validation/test split belum ada | Bagian 1: split video secara acak (level-video, bukan level-frame) jadi `train_videos` / `val_videos` / `test_videos` |
| 20 video training ikut dievaluasi (data leakage) | Bagian 8+ hanya mengevaluasi `test_videos`, yang **tidak pernah dilihat** saat training Neural Codec maupun Encoder-Decoder |
| Unseen watermark belum diuji | Bagian 9: evaluasi dengan payload teks yang **tidak pernah** jadi target eksplisit saat training |
| Retry mechanism belum tervalidasi | Bagian 10: kompresi dipaksa lebih agresif dari kondisi training supaya jalur "No" pada flowchart benar-benar teruji |
| H.264/H.265 hanya diuji 1 titik CRF | Bagian 11: pencarian breaking point di beberapa CRF (23 s.d. 46) |
| Compression efficiency (bitrate/ratio) belum diukur | Bagian 12: ukuran file, bitrate, dan rasio kompresi Neural Codec vs H.264/H.265 |

Alur inti pipeline (flowchart) tidak berubah dari versi sebelumnya:
```
Input Video -> Insert Watermark -> Compress Video (Neural Codec) -> Rescan Watermark
 -> legible? --No--> balik ke Insert Watermark (strength dinaikkan, maks. MAX_RETRY kali)
            --Yes--> print(True, biner payload)
```


## 1. Setup & Konfigurasi Global (+ Train/Val/Test Split)

Split dilakukan **di level video** (bukan level frame) dan **sebelum** video apa pun dipakai
untuk training apa pun — supaya `test_videos` benar-benar *unseen* di semua tahap berikutnya.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q opencv-python-headless scikit-image torch torchvision tqdm pandas matplotlib
!apt-get -y install ffmpeg > /dev/null 2>&1

print("Setup selesai. Google Drive berhasil terpasang di /content/drive")


In [ ]:
import os, random, gc, tempfile, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DATASET_DIR = '/content/drive/MyDrive/Video_data/test'
OUTPUT_DIR  = '/content/drive/MyDrive/Video_data/output'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'watermarked'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'compressed'), exist_ok=True)

# ------------------------------------------------------------
# PARAMETER UTAMA
# ------------------------------------------------------------
FRAME_SIZE            = (128, 128)
MAX_FRAMES_PER_VIDEO   = 100
SEED                   = 42

WATERMARK_TEXT = "sabila"          # watermark yang DIPAKAI SEBAGIAN saat training (target mixing)
ECC_REPEAT     = 3

MAX_RETRY            = 5
RETRY_STRENGTH_STEP  = 0.15

CRF_HIGH_COMPRESSION = 35           # dipakai H.264/H.265 opsional (Bagian di luar loop utama)

# --- Rasio Train / Val / Test (level video) ---
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15   # sisanya

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device yang digunakan:", device)

# ------------------------------------------------------------
# Daftar video dataset (kecualikan file hasil-generate sesi sebelumnya)
# ------------------------------------------------------------
_GENERATED_MARKERS = ('_watermarked', '_h264', '_h265', '_neuralcodec')

def _is_generated_output(filename):
    stem = os.path.splitext(filename)[0].lower()
    return any(marker in stem for marker in _GENERATED_MARKERS)

_all_video_files = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
video_list = sorted([f for f in _all_video_files if not _is_generated_output(f)])
print(f"Ditemukan {len(video_list)} video ASLI di DATASET_DIR.")

# ------------------------------------------------------------
# SPLIT LEVEL-VIDEO -- dilakukan SEBELUM training apa pun
# ------------------------------------------------------------
_rng = random.Random(SEED)
_shuffled = video_list.copy()
_rng.shuffle(_shuffled)

n_total = len(_shuffled)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)

train_videos = sorted(_shuffled[:n_train])
val_videos   = sorted(_shuffled[n_train:n_train + n_val])
test_videos  = sorted(_shuffled[n_train + n_val:])

assert set(train_videos).isdisjoint(val_videos)
assert set(train_videos).isdisjoint(test_videos)
assert set(val_videos).isdisjoint(test_videos)
assert len(train_videos) + len(val_videos) + len(test_videos) == n_total

print(f"\nSplit dataset (level video, seed={SEED}):")
print(f"  train_videos : {len(train_videos)} video ({len(train_videos)/n_total:.0%})")
print(f"  val_videos   : {len(val_videos)} video ({len(val_videos)/n_total:.0%})")
print(f"  test_videos  : {len(test_videos)} video ({len(test_videos)/n_total:.0%})  <-- TIDAK PERNAH dilihat saat training")

split_df = pd.concat([
    pd.DataFrame({'video': train_videos, 'split': 'train'}),
    pd.DataFrame({'video': val_videos, 'split': 'val'}),
    pd.DataFrame({'video': test_videos, 'split': 'test'}),
], ignore_index=True)
split_df.to_csv(os.path.join(OUTPUT_DIR, 'dataset_split.csv'), index=False)
print(f"\nDaftar split disimpan ke: {os.path.join(OUTPUT_DIR, 'dataset_split.csv')}")


## 2. Preprocessing Video (Ekstraksi Frame, Resize, Normalisasi)

Frame dimuat untuk **seluruh** video (train+val+test) sekali di awal supaya efisien -- tapi
INGAT: yang membedakan train/val/test adalah **video mana yang dipakai di tahap training vs
tahap evaluasi**, bukan proses ekstraksi framenya (proses ini sama untuk semua video).

In [ ]:
def extract_frames(video_path, frame_size=FRAME_SIZE, max_frames=MAX_FRAMES_PER_VIDEO):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frames = []
    count = 0
    while cap.isOpened() and count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (frame_size[1], frame_size[0]))
        frame = frame.astype(np.float32) / 255.0
        frames.append(frame)
        count += 1
    cap.release()
    if len(frames) == 0:
        raise RuntimeError(f"Tidak ada frame yang berhasil dibaca dari: {video_path}")
    return frames, fps, (orig_w, orig_h)


def frames_to_video_lossless(frames, out_path, fps, size=None):
    if size is None:
        h, w = frames[0].shape[:2]
    else:
        w, h = size
    with tempfile.TemporaryDirectory() as tmpdir:
        for i, f in enumerate(frames):
            f_uint8 = np.clip(f * 255.0, 0, 255).astype(np.uint8)
            f_uint8 = cv2.resize(f_uint8, (w, h))
            f_bgr = cv2.cvtColor(f_uint8, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(tmpdir, f'f_{i:05d}.png'), f_bgr)
        cmd = ['ffmpeg', '-y', '-framerate', str(fps),
               '-i', os.path.join(tmpdir, 'f_%05d.png'), '-c:v', 'ffv1', out_path]
        result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        if result.returncode != 0 or not os.path.exists(out_path):
            raise RuntimeError(f"ffmpeg gagal menulis {out_path}: {result.stderr.decode(errors='ignore')[-500:]}")
    return out_path


def compress_ffmpeg(input_path, output_path, codec='libx264', crf=32):
    cmd = ['ffmpeg', '-y', '-i', input_path, '-c:v', codec, '-crf', str(crf), '-preset', 'fast', output_path]
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    if result.returncode != 0 or not os.path.exists(output_path):
        raise RuntimeError(f"ffmpeg gagal kompresi {input_path}: {result.stderr.decode(errors='ignore')[-500:]}")
    return output_path


dataset_frames = {}
for vname in video_list:
    path = os.path.join(DATASET_DIR, vname)
    frames, fps, size = extract_frames(path)
    dataset_frames[vname] = (frames, fps, size)
print(f"[LOAD] {len(dataset_frames)} video berhasil dimuat ke memori.")


## 3. Payload & Error Correcting Code (ECC)

In [ ]:
PAYLOAD_BITS = 8 * len(WATERMARK_TEXT)
WATERMARK_LENGTH = PAYLOAD_BITS * ECC_REPEAT

def text_to_bits_ecc(text, repeat=ECC_REPEAT, bit_length=None):
    raw = []
    for ch in text:
        raw.extend([int(b) for b in format(ord(ch), '08b')])
    bits = raw * repeat
    if bit_length is None:
        bit_length = len(raw) * repeat
    bits = bits[:bit_length]
    bits += [0] * (bit_length - len(bits))
    return bits, len(raw)


def bits_to_text_ecc(bits, n_payload_bits=PAYLOAD_BITS, repeat=ECC_REPEAT):
    bits = [int(round(float(b))) for b in bits]
    chunks = [bits[i * n_payload_bits:(i + 1) * n_payload_bits] for i in range(repeat)]
    voted = []
    for j in range(n_payload_bits):
        votes = [chunks[k][j] for k in range(repeat) if j < len(chunks[k])]
        voted.append(1 if sum(votes) > len(votes) / 2 else 0)
    chars = []
    for i in range(0, len(voted) - len(voted) % 8, 8):
        byte = voted[i:i + 8]
        val = int(''.join(map(str, byte)), 2)
        if 32 <= val <= 126:
            chars.append(chr(val))
    return ''.join(chars)


wm_bits_list, _ = text_to_bits_ecc(WATERMARK_TEXT)
GROUND_TRUTH_WM = torch.tensor([wm_bits_list], dtype=torch.float32).to(device)

# Payload UNSEEN -- tidak pernah dijadikan target eksplisit saat training (Bagian 9).
# Panjang karakter sama dengan WATERMARK_TEXT supaya WATERMARK_LENGTH konsisten (arsitektur
# encoder/decoder punya ukuran output tetap, jadi payload uji harus sama panjang karakternya).
UNSEEN_WATERMARKS = ["skrpsi", "kamera", "budiar"]
assert all(len(w) == len(WATERMARK_TEXT) for w in UNSEEN_WATERMARKS), \
    "Panjang setiap UNSEEN_WATERMARKS harus sama dengan panjang WATERMARK_TEXT"

print(f"WATERMARK_TEXT (dipakai training) : '{WATERMARK_TEXT}'")
print(f"UNSEEN_WATERMARKS (uji generalisasi): {UNSEEN_WATERMARKS}")
print(f"PAYLOAD_BITS={PAYLOAD_BITS}  ECC_REPEAT={ECC_REPEAT}x  WATERMARK_LENGTH={WATERMARK_LENGTH}")
assert bits_to_text_ecc(wm_bits_list) == WATERMARK_TEXT, "Encode/decode ECC tidak konsisten!"


## 4. Arsitektur Model

In [ ]:
class WatermarkEncoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=80):
        super().__init__()
        self.wm_length = wm_length
        self.fc_wm = nn.Linear(wm_length, 16 * 8 * 8)
        self.wm_upsample = nn.Sequential(
            nn.ConvTranspose2d(16, 32, 4, stride=2, padding=1), nn.GroupNorm(8, 32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.GroupNorm(4, 16), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16, 8, 4, stride=2, padding=1), nn.GroupNorm(2, 8), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(8, 4, 4, stride=2, padding=1), nn.GroupNorm(2, 4), nn.ReLU(inplace=True),
        )
        self.conv_in = nn.Sequential(
            nn.Conv2d(3 + 4, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_mid = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_mid2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
        )
        self.conv_out = nn.Conv2d(channels, 3, 3, padding=1)

    def forward(self, frame, wm_bits, strength=0.18):
        wm_feat = self.fc_wm(wm_bits).view(wm_bits.size(0), 16, 8, 8)
        wm_map = self.wm_upsample(wm_feat)
        x = torch.cat([frame, wm_map], dim=1)
        x = self.conv_in(x)
        x = self.conv_mid(x) + x
        x = self.conv_mid2(x) + x
        residual = torch.tanh(self.conv_out(x)) * strength
        watermarked = torch.clamp(frame + residual, 0.0, 1.0)
        return watermarked, residual


class WatermarkDecoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=80):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, stride=2, padding=1), nn.GroupNorm(8, channels), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc = nn.Sequential(
            nn.Linear(channels * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Linear(256, wm_length),
        )

    def forward(self, frame):
        x = self.features(frame)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)


def ste_quantize(z, levels=16):
    z_hard = torch.round(z * levels) / levels
    return z + (z_hard - z).detach()


class NeuralCodec(nn.Module):
    def __init__(self, bottleneck=8):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, bottleneck, 3, padding=1),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(bottleneck, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, 3, padding=1),
        )

    def forward(self, x, quant_levels=16):
        z = torch.tanh(self.enc(x))
        z = ste_quantize(z, quant_levels)
        out = torch.sigmoid(self.dec(z))
        return out

encoder = WatermarkEncoder().to(device)
decoder = WatermarkDecoder().to(device)
neural_codec = NeuralCodec().to(device)
print("Arsitektur berhasil didefinisikan.")


## 4B. Fungsi Bantu Evaluasi (dipakai untuk monitoring validasi & stress test)

Fungsi in-memory (tanpa menulis file video) supaya cepat dipakai berulang kali: saat
memantau performa di `val_videos` selama training, maupun di stress test Bagian 10.

In [ ]:
@torch.no_grad()
def compute_bit_accuracy_in_memory(video_names, wm_text=WATERMARK_TEXT, quant_levels=16,
                                    apply_neural_codec=True, strength=0.18, max_videos=None):
    '''Sisipkan watermark -> (opsional) lewatkan Neural Codec -> decode, semua di memori
    (tanpa nulis file). Return dict: bit_accuracy rata-rata & legible_rate (per-video majority
    vote persis seperti alur rescan_watermark asli).'''
    encoder.eval(); decoder.eval(); neural_codec.eval()
    names = video_names if max_videos is None else video_names[:max_videos]
    target_bits, _ = text_to_bits_ecc(wm_text)
    target_t = torch.tensor([target_bits], dtype=torch.float32).to(device)

    total_correct, total_bits, legible_count = 0, 0, 0
    for vname in names:
        frames, fps, _ = dataset_frames[vname]
        frames_t = torch.tensor(np.stack(frames)).permute(0, 3, 1, 2).float().to(device)
        wm_batch = target_t.repeat(frames_t.size(0), 1)
        watermarked, _ = encoder(frames_t, wm_batch, strength=strength)
        noised = neural_codec(watermarked, quant_levels=quant_levels) if apply_neural_codec else watermarked
        logits = decoder(noised)
        probs = torch.sigmoid(logits)
        pred_bits_per_frame = (probs > 0.5).float()

        total_correct += (pred_bits_per_frame == wm_batch).sum().item()
        total_bits += wm_batch.numel()

        avg_probs = probs.mean(dim=0)
        extracted_text = bits_to_text_ecc((avg_probs > 0.5).float().cpu().numpy())
        if extracted_text == wm_text:
            legible_count += 1

        del frames_t, watermarked, noised, logits
    return {
        'bit_accuracy': total_correct / total_bits if total_bits else 0.0,
        'legible_rate': legible_count / len(names) if names else 0.0,
        'n_videos': len(names),
    }

print("Fungsi evaluasi in-memory siap dipakai.")


## 5. Training Neural Codec (HANYA memakai `train_videos`)

In [ ]:
MAX_VIDEOS_FOR_CODEC_TRAINING = 20
codec_train_videos = train_videos[:MAX_VIDEOS_FOR_CODEC_TRAINING]
print(f"Neural Codec dilatih dengan {len(codec_train_videos)} video dari TRAIN split (bukan seluruh dataset).")

CODEC_TRAIN_FRAMES = 300
CODEC_BATCH_SIZE = 8
NEURAL_CODEC_EPOCHS = 60

codec_frame_pool = []
for vname in codec_train_videos:
    frames, _, _ = dataset_frames[vname]
    codec_frame_pool.extend(frames)
    if len(codec_frame_pool) >= CODEC_TRAIN_FRAMES:
        break
codec_frame_pool = codec_frame_pool[:CODEC_TRAIN_FRAMES]

all_frames_t = torch.tensor(np.stack(codec_frame_pool)).permute(0, 3, 1, 2).float().to(device)
n_codec = all_frames_t.size(0)
print(f"Neural Codec akan dilatih dengan {n_codec} frame (dari video TRAIN saja).")

codec_opt = torch.optim.Adam(neural_codec.parameters(), lr=1e-3)
for epoch in range(NEURAL_CODEC_EPOCHS):
    perm = torch.randperm(n_codec)
    ep_loss = 0.0
    for i in range(0, n_codec, CODEC_BATCH_SIZE):
        idx = perm[i:i + CODEC_BATCH_SIZE]
        batch = all_frames_t[idx]
        recon = neural_codec(batch)
        loss = F.mse_loss(recon, batch)
        codec_opt.zero_grad(); loss.backward(); codec_opt.step()
        ep_loss += loss.item() * batch.size(0)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"[Neural Codec] Epoch {epoch+1}/{NEURAL_CODEC_EPOCHS} | recon_loss={ep_loss/n_codec:.5f}")

for p in neural_codec.parameters():
    p.requires_grad_(False)
neural_codec.eval()
torch.save(neural_codec.state_dict(), os.path.join(OUTPUT_DIR, 'neural_codec.pth'))
print("Neural Codec selesai dilatih & dibekukan.")


## 6. Training Encoder-Decoder (HANYA `train_videos`, val dipantau dari `val_videos`)

Perbedaan penting dari versi sebelumnya: kriteria *early stopping* sekarang dihitung dari
**`val_videos`** (video yang tidak pernah dipakai untuk update bobot), bukan dari batch training
itu sendiri — supaya angka `bit_acc` yang dipakai untuk berhenti training benar-benar mengukur
generalisasi, bukan hafalan.

In [ ]:
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
mse_loss = nn.MSELoss()
bce_loss = nn.BCEWithLogitsLoss()

MAX_VIDEOS_FOR_WM_TRAINING = 40
wm_train_videos = train_videos[:MAX_VIDEOS_FOR_WM_TRAINING]
print(f"Encoder-Decoder dilatih dengan {len(wm_train_videos)} video dari TRAIN split.")
print(f"Validasi selama training memakai {min(15, len(val_videos))} video dari VAL split (unseen).")

EPOCHS = 200
BATCH_SIZE = 8
WARMUP_EPOCHS = 5
CRF_CURRICULUM_EPOCHS = 120
TARGET_BIT_ACC_VAL = 0.92          # <-- kriteria berhenti sekarang diukur di VAL, bukan train
WM_TARGET_PROB = 0.5
VAL_EVERY = 10
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, 'checkpoint_wm.pth')
CHECKPOINT_EVERY = 10


def augment_batch(x):
    if torch.rand(1).item() < 0.5:
        x = torch.flip(x, dims=[3])
    return x


def ffmpeg_bpda_compress(x, fps=25, crf_range=(25, 38), codecs=('libx264',), preset='fast'):
    B, C, H, W = x.shape
    codec = random.choice(codecs)
    crf = random.randint(*crf_range)
    with tempfile.TemporaryDirectory() as tmpdir:
        frames_np = x.detach().permute(0, 2, 3, 1).cpu().numpy()
        png_dir = os.path.join(tmpdir, 'png')
        os.makedirs(png_dir, exist_ok=True)
        for i, f in enumerate(frames_np):
            f_uint8 = np.clip(f * 255.0, 0, 255).astype(np.uint8)
            cv2.imwrite(os.path.join(png_dir, f'f_{i:04d}.png'), cv2.cvtColor(f_uint8, cv2.COLOR_RGB2BGR))
        out_path = os.path.join(tmpdir, 'out.mp4')
        cmd = ['ffmpeg', '-y', '-framerate', str(fps), '-i', os.path.join(png_dir, 'f_%04d.png'),
               '-c:v', codec, '-crf', str(crf), '-preset', preset, '-pix_fmt', 'yuv420p', out_path]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        cap = cv2.VideoCapture(out_path)
        out_frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (W, H))
            out_frames.append(frame.astype(np.float32) / 255.0)
        cap.release()
        while len(out_frames) < B:
            out_frames.append(out_frames[-1] if out_frames else frames_np[len(out_frames)])
        out_frames = out_frames[:B]
    x_compressed = torch.tensor(np.stack(out_frames)).permute(0, 3, 1, 2).float().to(x.device)
    return x + (x_compressed - x).detach()


def apply_curriculum_noise(x, epoch, prev_val_bit_acc=0.5):
    if epoch <= WARMUP_EPOCHS:
        return x, 'none'
    progress = min((epoch - WARMUP_EPOCHS) / CRF_CURRICULUM_EPOCHS, 1.0)
    pool = ['gaussian', 'blur', 'quantize', 'neural']
    real_noise_ready = progress > 0.15 and (prev_val_bit_acc >= 0.55 or epoch > WARMUP_EPOCHS + 40)
    if real_noise_ready:
        pool += ['h264_real', 'h265_real']
    mode = random.choice(pool)

    if mode == 'gaussian':
        x = x + torch.randn_like(x) * (0.005 + 0.02 * progress)
    elif mode == 'blur' and progress > 0.3:
        x = F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)
    elif mode == 'quantize':
        levels = int(64 - 36 * progress)
        x = torch.round(x * levels) / levels
    elif mode == 'neural':
        x = neural_codec(x)
    elif mode in ('h264_real', 'h265_real'):
        codec = 'libx264' if mode == 'h264_real' else 'libx265'
        sub_progress = min(max((progress - 0.15) / 0.85, 0.0), 1.0)
        crf_lo = int(18 + 10 * sub_progress)
        crf_hi = int(28 + 10 * sub_progress)
        x = ffmpeg_bpda_compress(x, crf_range=(crf_lo, crf_hi), codecs=(codec,), preset='fast')
    return torch.clamp(x, 0.0, 1.0), mode


history = {'loss': [], 'bit_acc_train': [], 'val_epoch': [], 'val_bit_acc': [], 'val_legible_rate': []}
start_epoch = 1
prev_val_bit_acc = 0.5

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    encoder.load_state_dict(ckpt['encoder']); decoder.load_state_dict(ckpt['decoder'])
    optimizer.load_state_dict(ckpt['optimizer']); history = ckpt['history']
    start_epoch = ckpt['epoch'] + 1
    prev_val_bit_acc = history['val_bit_acc'][-1] if history['val_bit_acc'] else 0.5
    print(f"[RESUME] Melanjutkan dari epoch {start_epoch}.")
else:
    print("[RESUME] Tidak ada checkpoint -- training dimulai dari epoch 1.")


def save_checkpoint(epoch):
    torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(),
                'optimizer': optimizer.state_dict(), 'history': history, 'epoch': epoch}, CHECKPOINT_PATH)


for epoch in range(start_epoch, EPOCHS + 1):
    if epoch <= WARMUP_EPOCHS:
        LAMBDA_WM, LAMBDA_IMG, LAMBDA_BIAS = 1.0, 0.0, 0.0
    else:
        ramp = min((epoch - WARMUP_EPOCHS) / 8, 1.0)
        gate = max(0.1, min((prev_val_bit_acc - 0.55) / (0.85 - 0.55), 1.0)) if prev_val_bit_acc > 0.55 else 0.1
        LAMBDA_WM, LAMBDA_IMG, LAMBDA_BIAS = 1.0, 1.0 * ramp * gate, 5.0 * ramp * gate

    encoder.train(); decoder.train()
    epoch_loss, correct_bits, total_bits, total_samples = 0.0, 0, 0, 0

    for vname in wm_train_videos:
        frames, fps, size = dataset_frames[vname]
        frames_np = np.stack(frames)
        n_local = frames_np.shape[0]
        starts = list(range(0, max(n_local - BATCH_SIZE, 0) + 1, BATCH_SIZE)) or [0]
        random.shuffle(starts)

        for start in starts:
            idx_local = np.arange(start, min(start + BATCH_SIZE, n_local))
            batch = torch.tensor(frames_np[idx_local]).permute(0, 3, 1, 2).float().to(device)
            batch = augment_batch(batch)
            B = batch.size(0)

            if random.random() < WM_TARGET_PROB:
                wm_bits = GROUND_TRUTH_WM.repeat(B, 1)
            else:
                wm_bits = torch.randint(0, 2, (1, WATERMARK_LENGTH)).float().to(device).repeat(B, 1)

            watermarked, residual = encoder(batch, wm_bits)
            noised, noise_mode = apply_curriculum_noise(watermarked, epoch, prev_val_bit_acc=prev_val_bit_acc)
            pred_logits = decoder(noised)

            loss_img = mse_loss(watermarked, batch)
            loss_wm = bce_loss(pred_logits, wm_bits)
            loss_bias = residual.mean(dim=(2, 3)).pow(2).mean()
            loss = LAMBDA_IMG * loss_img + LAMBDA_WM * loss_wm + LAMBDA_BIAS * loss_bias

            optimizer.zero_grad(); loss.backward(); optimizer.step()

            with torch.no_grad():
                pred_bits = (torch.sigmoid(pred_logits) > 0.5).float()
                correct_bits += (pred_bits == wm_bits).sum().item()
                total_bits += wm_bits.numel()

            epoch_loss += loss.item() * B
            total_samples += B
            del batch, watermarked, noised, pred_logits
        del frames, frames_np

    gc.collect(); torch.cuda.empty_cache()

    bit_acc_train = correct_bits / total_bits
    avg_loss = epoch_loss / total_samples
    history['loss'].append(avg_loss); history['bit_acc_train'].append(bit_acc_train)

    tag = "[WARM-UP]" if epoch <= WARMUP_EPOCHS else ""
    log_line = f"Epoch {epoch:03d}/{EPOCHS} {tag} | loss={avg_loss:.5f} | bit_acc_train={bit_acc_train:.4f}"

    target_reached = False
    if epoch % VAL_EVERY == 0 or epoch == EPOCHS:
        val_result = compute_bit_accuracy_in_memory(
            val_videos, wm_text=WATERMARK_TEXT, quant_levels=16,
            apply_neural_codec=True, max_videos=min(15, len(val_videos)))
        prev_val_bit_acc = val_result['bit_accuracy']
        history['val_epoch'].append(epoch)
        history['val_bit_acc'].append(val_result['bit_accuracy'])
        history['val_legible_rate'].append(val_result['legible_rate'])
        log_line += f" | VAL(unseen)_bit_acc={val_result['bit_accuracy']:.4f} | VAL_legible_rate={val_result['legible_rate']:.2f}"
        target_reached = (val_result['bit_accuracy'] >= TARGET_BIT_ACC_VAL and epoch >= WARMUP_EPOCHS + VAL_EVERY)

    print(log_line)

    if epoch % CHECKPOINT_EVERY == 0:
        save_checkpoint(epoch)

    curriculum_full = (epoch - WARMUP_EPOCHS) >= CRF_CURRICULUM_EPOCHS
    if target_reached and curriculum_full:
        print(f"\nTarget VAL bit_acc {TARGET_BIT_ACC_VAL:.0%} tercapai (diukur di video UNSEEN) pada epoch {epoch}. Training dihentikan.")
        save_checkpoint(epoch)
        break
else:
    save_checkpoint(EPOCHS)

torch.save(encoder.state_dict(), os.path.join(OUTPUT_DIR, 'encoder.pth'))
torch.save(decoder.state_dict(), os.path.join(OUTPUT_DIR, 'decoder.pth'))
print("Model encoder/decoder tersimpan.")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.plot(history['loss']); plt.title('Loss (train)')
plt.subplot(1, 2, 2)
plt.plot(history['bit_acc_train'], label='bit_acc TRAIN', alpha=0.5)
plt.plot(history['val_epoch'], history['val_bit_acc'], 'o-', label='bit_acc VAL (unseen)', color='green')
plt.axhline(TARGET_BIT_ACC_VAL, color='r', linestyle='--', label='target VAL')
plt.legend(fontsize=8); plt.title('Bit Accuracy: Train vs Val (unseen)'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curve.png'), dpi=150); plt.show()


## 7. PIPELINE UTAMA (sesuai flowchart)

Sama seperti sebelumnya, tiap fungsi = satu kotak flowchart. Ditambah parameter `wm_text` dan
`quant_levels` supaya bisa dipakai ulang untuk uji unseen-watermark (Bagian 9) dan stress test
kompresi (Bagian 10).

In [ ]:
encoder.eval(); decoder.eval(); neural_codec.eval()


def insert_watermark(video_name, wm_text=WATERMARK_TEXT, strength=0.18):
    '''[Insert Watermark]'''
    frames, fps, orig_size = dataset_frames[video_name]
    frames_t = torch.tensor(np.stack(frames)).permute(0, 3, 1, 2).float().to(device)
    target_bits, _ = text_to_bits_ecc(wm_text)
    target_t = torch.tensor([target_bits], dtype=torch.float32).to(device)
    wm_batch = target_t.repeat(frames_t.size(0), 1)
    with torch.no_grad():
        watermarked_t, _ = encoder(frames_t, wm_batch, strength=strength)
    watermarked_np = watermarked_t.permute(0, 2, 3, 1).cpu().numpy()

    out_name = os.path.splitext(video_name)[0] + '_watermarked.mkv'
    out_path = os.path.join(OUTPUT_DIR, 'watermarked', out_name)
    frames_to_video_lossless(list(watermarked_np), out_path, fps, size=FRAME_SIZE[::-1])
    return out_path


def compress_neural_codec(video_path, video_name, quant_levels=16, tag='neuralcodec'):
    '''[Compress Video] quant_levels lebih kecil = kompresi lebih agresif (dipakai Bagian 10).'''
    frames, fps, _ = extract_frames(video_path)
    frames_t = torch.tensor(np.stack(frames)).permute(0, 3, 1, 2).float().to(device)
    with torch.no_grad():
        recon = neural_codec(frames_t, quant_levels=quant_levels)
    recon_np = recon.permute(0, 2, 3, 1).cpu().numpy()

    out_name = os.path.splitext(video_name)[0] + f'_{tag}.mkv'
    out_path = os.path.join(OUTPUT_DIR, 'compressed', out_name)
    frames_to_video_lossless(list(recon_np), out_path, fps, size=FRAME_SIZE[::-1])
    return out_path


def rescan_watermark(video_path):
    '''[Rescan Watermark]'''
    frames, fps, _ = extract_frames(video_path)
    frames_t = torch.tensor(np.stack(frames)).permute(0, 3, 1, 2).float().to(device)
    with torch.no_grad():
        probs = torch.sigmoid(decoder(frames_t))
    avg_probs = probs.mean(dim=0)
    extracted_bits = (avg_probs > 0.5).float().cpu().numpy()
    extracted_text = bits_to_text_ecc(extracted_bits)
    return extracted_bits, extracted_text


def is_legible(extracted_text, target_text):
    '''[is legible Watermark?]'''
    return extracted_text == target_text


def run_watermark_pipeline(video_name, wm_text=WATERMARK_TEXT, quant_levels=16,
                            max_retry=None, verbose=True):
    if max_retry is None:
        max_retry = MAX_RETRY
    attempt = 1
    extracted_bits, extracted_text = None, None
    wm_path, comp_path = None, None
    while attempt <= max_retry:
        strength = 0.18 * (1.0 + RETRY_STRENGTH_STEP * (attempt - 1))
        if verbose:
            print(f"[{video_name}] Percobaan {attempt}/{max_retry} (strength={strength:.3f}, quant_levels={quant_levels})")
        wm_path = insert_watermark(video_name, wm_text=wm_text, strength=strength)
        comp_path = compress_neural_codec(wm_path, video_name, quant_levels=quant_levels)
        extracted_bits, extracted_text = rescan_watermark(comp_path)

        if is_legible(extracted_text, wm_text):
            if verbose:
                print(f"  -> LEGIBLE: '{extracted_text}'")
            return {'video': video_name, 'status': True, 'attempts': attempt, 'wm_text': wm_text,
                    'extracted_text': extracted_text, 'payload_bits': extracted_bits.astype(int).tolist(),
                    'watermarked_path': wm_path, 'compressed_path': comp_path}
        else:
            if verbose:
                print(f"  -> TIDAK legible: '{extracted_text}' (target '{wm_text}')")
            attempt += 1

    return {'video': video_name, 'status': False, 'attempts': max_retry, 'wm_text': wm_text,
            'extracted_text': extracted_text,
            'payload_bits': extracted_bits.astype(int).tolist() if extracted_bits is not None else None,
            'watermarked_path': wm_path, 'compressed_path': comp_path}

print(f"Pipeline siap. MAX_RETRY={MAX_RETRY}")


## 8. Evaluasi Utama — HANYA pada `test_videos` (unseen), watermark `"sabila"`

Ini memperbaiki temuan *data leakage* sebelumnya: video di sini **tidak pernah** dipakai untuk
training Neural Codec maupun Encoder-Decoder (lihat Bagian 1, 5, 6).

In [ ]:
print(f"Mengevaluasi {len(test_videos)} video dari TEST split (tidak pernah dilihat saat training)...\n")

pipeline_results = []
for vname in test_videos:
    result = run_watermark_pipeline(vname, wm_text=WATERMARK_TEXT, quant_levels=16, verbose=False)
    pipeline_results.append(result)

df_pipeline = pd.DataFrame([{
    'video': r['video'], 'status': r['status'], 'attempts': r['attempts'],
    'extracted_text': r['extracted_text'],
    'payload_bits': ''.join(map(str, r['payload_bits'])) if r['payload_bits'] else None,
} for r in pipeline_results])
df_pipeline.to_csv(os.path.join(OUTPUT_DIR, 'hasil_pipeline_test_unseen.csv'), index=False)
display(df_pipeline)

n_success = df_pipeline['status'].sum()
print(f"\nHasil pada TEST SET (unseen, n={len(df_pipeline)}): {n_success}/{len(df_pipeline)} berhasil "
      f"({n_success/len(df_pipeline):.1%}).")
print("Angka ini adalah estimasi generalisasi yang valid, karena video test tidak pernah dipakai saat training.")


## 9. Uji Unseen Watermark — payload yang tidak pernah jadi target eksplisit saat training

Dijalankan di `test_videos` yang sama, tapi memakai teks watermark dari `UNSEEN_WATERMARKS`
(Bagian 3) — bukan `"sabila"`.

In [ ]:
MAX_TEST_VIDEOS_FOR_UNSEEN_WM = min(30, len(test_videos))
unseen_wm_results = []
for wm_text in UNSEEN_WATERMARKS:
    for vname in test_videos[:MAX_TEST_VIDEOS_FOR_UNSEEN_WM]:
        result = run_watermark_pipeline(vname, wm_text=wm_text, quant_levels=16, verbose=False)
        unseen_wm_results.append(result)

df_unseen_wm = pd.DataFrame([{
    'video': r['video'], 'wm_text': r['wm_text'], 'status': r['status'],
    'attempts': r['attempts'], 'extracted_text': r['extracted_text'],
} for r in unseen_wm_results])
df_unseen_wm.to_csv(os.path.join(OUTPUT_DIR, 'hasil_unseen_watermark.csv'), index=False)

summary_unseen = df_unseen_wm.groupby('wm_text')['status'].mean().rename('success_rate')
print("=== Success rate per payload UNSEEN (di TEST SET) ===")
display(summary_unseen.to_frame())

baseline_rate = n_success / len(df_pipeline)
print(f"\nPembanding: success rate watermark 'SEEN' ({WATERMARK_TEXT}) di test set = {baseline_rate:.1%}")
print("Kalau success rate UNSEEN mendekati angka SEEN di atas, ini bukti sistem menggeneralisasi payload,")
print("bukan menghafal pola bit 'sabila' secara spesifik.")


## 10. Stress Test Retry Mechanism

Neural Codec dipaksa jauh lebih agresif (`quant_levels` diperkecil, dari default training=16
menjadi 4) dibanding kondisi saat training, supaya attempt pertama **realistis punya peluang
gagal** — baru di situ jalur "No" pada flowchart benar-benar teruji.

In [ ]:
STRESS_QUANT_LEVELS = 4     # jauh lebih agresif dari training (16) -> attempt-1 dipaksa lebih sulit
MAX_TEST_VIDEOS_FOR_STRESS = min(20, len(test_videos))

stress_results = []
for vname in test_videos[:MAX_TEST_VIDEOS_FOR_STRESS]:
    result = run_watermark_pipeline(vname, wm_text=WATERMARK_TEXT, quant_levels=STRESS_QUANT_LEVELS, verbose=False)
    stress_results.append(result)

df_stress = pd.DataFrame([{
    'video': r['video'], 'status': r['status'], 'attempts_dipakai': r['attempts'],
} for r in stress_results])
df_stress.to_csv(os.path.join(OUTPUT_DIR, 'hasil_stress_retry.csv'), index=False)
display(df_stress)

n_attempt1 = (df_stress['attempts_dipakai'] == 1).sum()
n_needed_retry = ((df_stress['attempts_dipakai'] > 1) & (df_stress['status'])).sum()
n_failed_all = (~df_stress['status']).sum()
n_success_total = df_stress['status'].sum()

print(f"\nDari {len(df_stress)} video (quant_levels={STRESS_QUANT_LEVELS}, jauh lebih agresif dari training):")
print(f"  - Langsung legible di attempt-1      : {n_attempt1}")
print(f"  - Baru legible SETELAH retry (>1x)    : {n_needed_retry}  <-- ini bukti retry mechanism berfungsi")
print(f"  - Gagal terus sampai MAX_RETRY habis   : {n_failed_all}")
print(f"  - Total berhasil (attempt-1 + retry)   : {n_success_total}/{len(df_stress)}")

if n_needed_retry == 0:
    print("\n[CATATAN] Belum ada kasus yang butuh retry pada quant_levels ini -- coba turunkan")
    print("STRESS_QUANT_LEVELS lagi (mis. 2) untuk memaksa attempt-1 lebih sering gagal.")


## 11. Breaking Point H.264 / H.265 — pengujian di beberapa titik CRF

Dijalankan pada video `test_videos` yang **sudah legible** di Bagian 8 (watermark tersisip via
Neural Codec), lalu masing-masing dikompresi ulang dengan H.264/H.265 di beberapa level CRF untuk
mencari titik di mana watermark mulai tidak terbaca. **Di luar loop utama** — tidak memengaruhi
status/retry pipeline inti.

In [ ]:
CRF_LEVELS = [23, 28, 32, 35, 38, 42, 46]
MAX_TEST_VIDEOS_FOR_CRF_SWEEP = min(15, len(test_videos))

legible_test_results = [r for r in pipeline_results if r['status']][:MAX_TEST_VIDEOS_FOR_CRF_SWEEP]

def bit_error_rate(bits_true, bits_pred):
    bits_true = np.array(bits_true).flatten(); bits_pred = np.array(bits_pred).flatten()
    return float(np.mean(bits_true != bits_pred))

gt_bits = np.array(text_to_bits_ecc(WATERMARK_TEXT)[0])
crf_sweep_rows = []
for r in legible_test_results:
    vname, wm_path = r['video'], r['watermarked_path']
    base = os.path.splitext(vname)[0]
    for codec_name, codec_lib in [('H.264', 'libx264'), ('H.265', 'libx265')]:
        for crf in CRF_LEVELS:
            out_path = os.path.join(OUTPUT_DIR, 'compressed', f'{base}_{codec_name.replace(".", "")}_crf{crf}.mp4')
            compress_ffmpeg(wm_path, out_path, codec=codec_lib, crf=crf)
            bits_pred, text_pred = rescan_watermark(out_path)
            ber = bit_error_rate(gt_bits, bits_pred)
            crf_sweep_rows.append({'Video': vname, 'Codec': codec_name, 'CRF': crf,
                                    'BER': round(ber, 4), 'Legible': text_pred == WATERMARK_TEXT})

df_crf_sweep = pd.DataFrame(crf_sweep_rows)
df_crf_sweep.to_csv(os.path.join(OUTPUT_DIR, 'crf_breaking_point.csv'), index=False)

legible_rate_by_crf = df_crf_sweep.groupby(['Codec', 'CRF'])['Legible'].mean().unstack('Codec')
print("=== Legible rate per CRF (semakin tinggi CRF = kompresi semakin agresif) ===")
display(legible_rate_by_crf)

plt.figure(figsize=(7, 4))
for codec_name in legible_rate_by_crf.columns:
    plt.plot(legible_rate_by_crf.index, legible_rate_by_crf[codec_name], 'o-', label=codec_name)
plt.xlabel('CRF (kompresi makin agresif ->)'); plt.ylabel('Legible rate')
plt.title('Breaking point H.264 vs H.265'); plt.legend(); plt.ylim(-0.05, 1.05)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'crf_breaking_point.png'), dpi=150); plt.show()

for codec_name in legible_rate_by_crf.columns:
    below_100 = legible_rate_by_crf[codec_name][legible_rate_by_crf[codec_name] < 1.0]
    breaking_crf = below_100.index.min() if len(below_100) else None
    print(f"{codec_name}: mulai gagal pada CRF >= {breaking_crf}" if breaking_crf is not None
          else f"{codec_name}: masih 100% legible di semua CRF yang diuji ({CRF_LEVELS})")


## 12. Compression Efficiency — ukuran file, bitrate, rasio kompresi

Mengukur klaim inti judul skripsi ("tahan kompresi TINGGI"): seberapa kecil Neural Codec
benar-benar menekan ukuran file dibanding video ber-watermark lossless, dan dibandingkan dengan
H.264/H.265 pada CRF yang representatif (`CRF_HIGH_COMPRESSION`, Bagian 1).

In [ ]:
MAX_TEST_VIDEOS_FOR_EFFICIENCY = min(15, len(legible_test_results))
efficiency_rows = []

for r in legible_test_results[:MAX_TEST_VIDEOS_FOR_EFFICIENCY]:
    vname, wm_path, nc_path = r['video'], r['watermarked_path'], r['compressed_path']
    base = os.path.splitext(vname)[0]
    frames, fps, _ = dataset_frames[vname]
    duration_sec = len(frames) / fps

    h264_path = os.path.join(OUTPUT_DIR, 'compressed', f'{base}_h264_eff.mp4')
    h265_path = os.path.join(OUTPUT_DIR, 'compressed', f'{base}_h265_eff.mp4')
    compress_ffmpeg(wm_path, h264_path, codec='libx264', crf=CRF_HIGH_COMPRESSION)
    compress_ffmpeg(wm_path, h265_path, codec='libx265', crf=CRF_HIGH_COMPRESSION)

    lossless_size = os.path.getsize(wm_path)
    for codec_name, path in [('Neural Codec', nc_path), ('H.264', h264_path), ('H.265', h265_path)]:
        size_bytes = os.path.getsize(path)
        bitrate_kbps = (size_bytes * 8 / 1000) / duration_sec
        compression_ratio = lossless_size / size_bytes
        efficiency_rows.append({
            'Video': vname, 'Metode': codec_name,
            'Ukuran File (KB)': round(size_bytes / 1024, 1),
            'Bitrate (kbps)': round(bitrate_kbps, 1),
            'Rasio Kompresi (vs lossless)': round(compression_ratio, 2),
        })

df_efficiency = pd.DataFrame(efficiency_rows)
df_efficiency.to_csv(os.path.join(OUTPUT_DIR, 'compression_efficiency.csv'), index=False)
print("=== Efisiensi kompresi per metode (rata-rata seluruh video yang diuji) ===")
display(df_efficiency)

summary_efficiency = df_efficiency.groupby('Metode')[['Ukuran File (KB)', 'Bitrate (kbps)', 'Rasio Kompresi (vs lossless)']].mean().round(2)
print("\n=== Ringkasan rata-rata ===")
display(summary_efficiency)

plt.figure(figsize=(7, 4))
plt.bar(summary_efficiency.index, summary_efficiency['Rasio Kompresi (vs lossless)'],
        color=['#55A868', '#4C72B0', '#DD8452'])
plt.ylabel('Rasio kompresi (semakin besar = semakin menekan ukuran)')
plt.title('Rasio Kompresi: Neural Codec vs H.264/H.265')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'compression_efficiency.png'), dpi=150); plt.show()


## 13. Kesimpulan

**Status setiap temuan dosen setelah revisi ini:**

| Temuan dosen | Status setelah revisi |
|---|---|
| Train/val/test split | ✅ Split level-video di Bagian 1, dipakai konsisten di seluruh notebook |
| Unseen test data | ✅ Bagian 8 hanya evaluasi `test_videos`, tidak pernah dilihat saat training (Bagian 5 & 6 hanya pakai `train_videos`, early stopping diukur di `val_videos`) |
| Unseen watermark | ✅ Bagian 9, payload di `UNSEEN_WATERMARKS` dibandingkan langsung dengan success rate payload yang dipakai saat training |
| Retry mechanism belum tervalidasi | ✅ Bagian 10, kompresi dipaksa lebih agresif dari training supaya jalur "No" pada flowchart benar-benar teruji dan retry-nya terbukti membantu |
| H.264/H.265 hanya 1 titik CRF | ✅ Bagian 11, breaking point dicari di 7 level CRF (23-46), terpisah per codec |
| Compression efficiency belum diukur | ✅ Bagian 12, ukuran file/bitrate/rasio kompresi Neural Codec vs H.264/H.265 pada CRF yang setara |

**Yang masih perlu diperhatikan untuk laporan skripsi:**
- Jumlah video di `test_videos`/`val_videos` bergantung total dataset (15% dari 225 video ≈ 33-34
  video per split) — kalau dosen minta test set lebih besar, ubah `TEST_RATIO` di Bagian 1.
- `MAX_VIDEOS_FOR_WM_TRAINING=40` dan berbagai `max_videos`/`MAX_TEST_VIDEOS_FOR_*` di tiap bagian
  membatasi jumlah video demi kecepatan running di Colab — kalau waktu eksekusi tidak masalah,
  angka-angka ini bisa dinaikkan (idealnya evaluasi Bagian 8-12 memakai seluruh `test_videos`).
- `STRESS_QUANT_LEVELS=4` di Bagian 10 adalah nilai awal; kalau retry masih belum pernah
  terpakai (`n_needed_retry==0`), turunkan nilainya (mis. 2) sampai attempt-1 benar-benar gagal
  pada sebagian video.
